# 1A TS-ResNet50 120k Modal All-in-One

Upload `LSWMD.pkl` to Modal, then run this notebook top to bottom.

This notebook is self-contained and will:
- find the uploaded raw pickle automatically
- build the `120k / 10k / 20k` labeled split if it is missing
- train the TS-ResNet50 teacher-student model with tqdm progress
- evaluate multiple score variants without retraining
- save checkpoints, metrics, CSVs, and defect breakdowns under `/output/ts_resnet50_120k_modal`


In [ ]:
from __future__ import annotations

import json
import math
import pickle
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas.core.indexes as core_indexes
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm

pd.options.display.max_rows = 200
pd.options.display.max_columns = 200
torch.backends.cudnn.benchmark = True

CONFIG = {
    "run": {
        "output_dir": "/output/ts_resnet50_120k_modal",
        "seed": 42,
        "rebuild_dataset": False,
        "resume_if_available": True,
        "run_training": True,
    },
    "data": {
        "raw_pickle_name": "LSWMD.pkl",
        "image_size": 64,
        "train_total": 120000,
        "train_anomalies": 6000,
        "val_total": 10000,
        "val_anomalies": 500,
        "test_total": 20000,
        "test_anomalies": 1000,
        "batch_size": 32,
        "num_workers": 2,
        "split_seed": 42,
        "candidate_search_dirs": [".", "/root", "/mnt/data", "/workspace", "/tmp", "/content", "/data"],
    },
    "training": {
        "device": "auto",
        "epochs": 30,
        "learning_rate": 3e-4,
        "weight_decay": 1e-5,
        "early_stopping_patience": 5,
        "early_stopping_min_delta": 1e-4,
        "checkpoint_every": 5,
    },
    "model": {
        "teacher_backbone": "resnet50",
        "teacher_layer": "layer2",
        "teacher_pretrained": True,
        "teacher_input_size": 224,
        "normalize_teacher_input": True,
        "feature_autoencoder_hidden_dim": 128,
        "student_weight": 1.0,
        "autoencoder_weight": 1.0,
        "score_student_weight": 1.0,
        "score_autoencoder_weight": 0.0,
        "reduction": "topk_mean",
        "topk_ratio": 0.20,
    },
    "scoring": {
        "threshold_quantile": 0.95,
        "variants": [
            {"name": "default", "score_student_weight": 1.0, "score_autoencoder_weight": 0.0, "reduction": "topk_mean", "topk_ratio": 0.20},
            {"name": "s1_a1_topk_mean_r020", "score_student_weight": 1.0, "score_autoencoder_weight": 1.0, "reduction": "topk_mean", "topk_ratio": 0.20},
            {"name": "s2_a1_topk_mean_r020", "score_student_weight": 2.0, "score_autoencoder_weight": 1.0, "reduction": "topk_mean", "topk_ratio": 0.20},
            {"name": "s2_a1_topk_mean_r015", "score_student_weight": 2.0, "score_autoencoder_weight": 1.0, "reduction": "topk_mean", "topk_ratio": 0.15},
            {"name": "s2_a1_mean", "score_student_weight": 2.0, "score_autoencoder_weight": 1.0, "reduction": "mean", "topk_ratio": 0.10},
            {"name": "s2_a1_max", "score_student_weight": 2.0, "score_autoencoder_weight": 1.0, "reduction": "max", "topk_ratio": 0.10},
        ],
    },
}

LABEL_NORMAL = "none"
LABEL_DEFECT = "pattern"

OUTPUT_DIR = Path(CONFIG["run"]["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_ROOT = OUTPUT_DIR / "prepared_120k"
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_SLUG = (
    f"train{CONFIG['data']['train_total']}_a{CONFIG['data']['train_anomalies']}"
    f"_val{CONFIG['data']['val_total']}_a{CONFIG['data']['val_anomalies']}"
    f"_test{CONFIG['data']['test_total']}_a{CONFIG['data']['test_anomalies']}"
)
METADATA_PATH = PREPARED_ROOT / f"metadata_{METADATA_SLUG}.csv"
ARRAYS_DIR = PREPARED_ROOT / f"arrays_{METADATA_SLUG}"
EVAL_DIR = OUTPUT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)


set_seed(int(CONFIG["run"]["seed"]))
DEVICE = resolve_device(str(CONFIG["training"]["device"]))
print(
    json.dumps(
        {
            "output_dir": str(OUTPUT_DIR),
            "prepared_root": str(PREPARED_ROOT),
            "device": str(DEVICE),
            "torch_cuda_available": torch.cuda.is_available(),
            "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "batch_size": int(CONFIG["data"]["batch_size"]),
        },
        indent=2,
    )
)


In [ ]:
def read_legacy_pickle(path: str | Path) -> pd.DataFrame:
    sys.modules["pandas.indexes"] = core_indexes
    with Path(path).open("rb") as handle:
        return pickle.load(handle, encoding="latin1")


def unwrap_legacy_value(value: Any) -> str:
    if value is None:
        return ""
    if hasattr(value, "size") and getattr(value, "size") == 0:
        return ""
    if hasattr(value, "tolist"):
        value = value.tolist()
    while isinstance(value, list) and len(value) == 1:
        value = value[0]
    return str(value).strip()


def find_uploaded_pickle(config: dict[str, Any]) -> Path:
    preferred_name = str(config["data"].get("raw_pickle_name", "LSWMD.pkl"))
    candidates: list[Path] = []
    cwd = Path.cwd().resolve()
    search_dirs = [cwd, *cwd.parents]
    for raw_dir in config["data"].get("candidate_search_dirs", []):
        path = Path(raw_dir)
        if not path.is_absolute():
            path = (cwd / path).resolve()
        if path.exists():
            search_dirs.append(path)

    seen_dirs: set[Path] = set()
    unique_dirs: list[Path] = []
    for path in search_dirs:
        try:
            resolved = path.resolve()
        except Exception:
            continue
        if resolved not in seen_dirs:
            seen_dirs.add(resolved)
            unique_dirs.append(resolved)

    for directory in unique_dirs:
        direct = directory / preferred_name
        if direct.exists():
            return direct
        candidates.extend(sorted(directory.glob("*.pkl")))
        candidates.extend(sorted(directory.glob("**/*.pkl")))

    seen_files: set[Path] = set()
    unique_files: list[Path] = []
    for path in candidates:
        try:
            resolved = path.resolve()
        except Exception:
            continue
        if resolved.is_file() and resolved not in seen_files:
            seen_files.add(resolved)
            unique_files.append(resolved)

    named_matches = [path for path in unique_files if path.name == preferred_name]
    if named_matches:
        return named_matches[0]
    if unique_files:
        return max(unique_files, key=lambda path: path.stat().st_size)

    raise FileNotFoundError("Could not find an uploaded pickle. Upload LSWMD.pkl next to the notebook or under /mnt/data, /root, /workspace, or /tmp.")


def normalize_map(wafer_map: np.ndarray, image_size: int) -> np.ndarray:
    wafer_map = np.asarray(wafer_map, dtype=np.float32)
    if wafer_map.ndim != 2:
        raise ValueError(f"Expected 2D wafer map, got shape {wafer_map.shape}")
    wafer_map = wafer_map / 2.0
    tensor = torch.from_numpy(wafer_map).unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(tensor, size=(image_size, image_size), mode="nearest")
    return resized.squeeze(0).squeeze(0).numpy()


def infer_label_from_row(row: pd.Series) -> str | None:
    failure = unwrap_legacy_value(row.get("failureType", "")).lower()
    if failure and failure != "none":
        return LABEL_DEFECT
    if failure == "none":
        return LABEL_NORMAL
    return None


def validate_labeled_split_counts(config: dict[str, Any], normal_count: int, defect_count: int) -> None:
    requested_normals = (
        int(config["data"]["train_total"]) - int(config["data"]["train_anomalies"])
        + int(config["data"]["val_total"]) - int(config["data"]["val_anomalies"])
        + int(config["data"]["test_total"]) - int(config["data"]["test_anomalies"])
    )
    requested_defects = (
        int(config["data"]["train_anomalies"])
        + int(config["data"]["val_anomalies"])
        + int(config["data"]["test_anomalies"])
    )
    if requested_normals > normal_count:
        raise ValueError(f"Requested {requested_normals} normals but only {normal_count} are available.")
    if requested_defects > defect_count:
        raise ValueError(f"Requested {requested_defects} defects but only {defect_count} are available.")


def build_labeled_split_dataframe(df: pd.DataFrame, config: dict[str, Any]) -> pd.DataFrame:
    seed = int(config["data"]["split_seed"])
    normal_df = df[df["label"] == LABEL_NORMAL].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    defect_df = df[df["label"] == LABEL_DEFECT].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    validate_labeled_split_counts(config, len(normal_df), len(defect_df))

    train_normals = int(config["data"]["train_total"]) - int(config["data"]["train_anomalies"])
    val_normals = int(config["data"]["val_total"]) - int(config["data"]["val_anomalies"])
    test_normals = int(config["data"]["test_total"]) - int(config["data"]["test_anomalies"])
    train_anomalies = int(config["data"]["train_anomalies"])
    val_anomalies = int(config["data"]["val_anomalies"])
    test_anomalies = int(config["data"]["test_anomalies"])

    train_normal_df = normal_df.iloc[:train_normals].copy()
    val_normal_df = normal_df.iloc[train_normals : train_normals + val_normals].copy()
    test_normal_df = normal_df.iloc[train_normals + val_normals : train_normals + val_normals + test_normals].copy()
    train_defect_df = defect_df.iloc[:train_anomalies].copy()
    val_defect_df = defect_df.iloc[train_anomalies : train_anomalies + val_anomalies].copy()
    test_defect_df = defect_df.iloc[train_anomalies + val_anomalies : train_anomalies + val_anomalies + test_anomalies].copy()

    train_normal_df["split"] = "train"
    val_normal_df["split"] = "val"
    test_normal_df["split"] = "test"
    train_defect_df["split"] = "train"
    val_defect_df["split"] = "val"
    test_defect_df["split"] = "test"
    return pd.concat(
        [train_normal_df, train_defect_df, val_normal_df, val_defect_df, test_normal_df, test_defect_df],
        ignore_index=True,
    )


class PreparedWaferDataset(Dataset):
    def __init__(self, metadata_df: pd.DataFrame, split: str) -> None:
        self.metadata = metadata_df[metadata_df["split"] == split].reset_index(drop=True).copy()

    def __len__(self) -> int:
        return len(self.metadata)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.metadata.iloc[index]
        wafer_map = np.load(row["array_path"]).astype(np.float32)
        tensor = torch.from_numpy(wafer_map).unsqueeze(0)
        label = torch.tensor(int(row["is_anomaly"]), dtype=torch.long)
        return tensor, label


def prepare_dataset(config: dict[str, Any], *, rebuild: bool = False) -> pd.DataFrame:
    if METADATA_PATH.exists() and not rebuild:
        metadata = pd.read_csv(METADATA_PATH)
        print(f"Loaded cached dataset from {METADATA_PATH}")
        return metadata

    pickle_path = find_uploaded_pickle(config)
    print(f"Using raw pickle: {pickle_path}")
    ARRAYS_DIR.mkdir(parents=True, exist_ok=True)

    df = read_legacy_pickle(pickle_path)
    df = df.copy()
    df["failureTypeText"] = df["failureType"].map(unwrap_legacy_value)
    df["trianTestLabelText"] = df["trianTestLabel"].map(unwrap_legacy_value)
    df["label"] = df.apply(infer_label_from_row, axis=1)
    df = df[df["label"].notna()].reset_index(drop=True)

    export_df = build_labeled_split_dataframe(df, config)
    records: list[dict[str, object]] = []
    iterator = tqdm(export_df.iterrows(), total=len(export_df), desc="prepare_dataset", leave=True)
    for row_index, row in iterator:
        file_name = f"wafer_{row_index:07d}.npy"
        array_path = ARRAYS_DIR / file_name
        raw_map = np.asarray(row["waferMap"])
        wafer_map = normalize_map(raw_map, image_size=int(config["data"]["image_size"]))
        np.save(array_path, wafer_map)
        records.append(
            {
                "array_path": str(array_path.resolve()),
                "label": row["label"],
                "defect_type": row["failureTypeText"] or "unlabeled",
                "is_anomaly": int(row["label"] == LABEL_DEFECT),
                "split": row["split"],
                "source_split": row["trianTestLabelText"] or "unlabeled",
                "original_height": int(raw_map.shape[0]),
                "original_width": int(raw_map.shape[1]),
            }
        )
        iterator.set_postfix(split=str(row["split"]))

    metadata = pd.DataFrame(records)
    metadata.to_csv(METADATA_PATH, index=False)
    split_summary = (
        metadata.groupby(["split", "is_anomaly"]).size().rename("count").reset_index().sort_values(["split", "is_anomaly"])
    )
    split_summary.to_csv(PREPARED_ROOT / "split_summary.csv", index=False)
    print(f"Saved dataset metadata to {METADATA_PATH}")
    display(split_summary)
    return metadata


In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score

"""Per-sample scoring utilities shared by training and evaluation code."""


import math

import torch
import torch.nn.functional as F


def reconstruction_mse(inputs: torch.Tensor, reconstructions: torch.Tensor) -> torch.Tensor:
    dims = tuple(range(1, inputs.ndim))
    return torch.mean((reconstructions - inputs).pow(2), dim=dims)


def normalized_kl_divergence(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)


def vae_anomaly_score(
    inputs: torch.Tensor,
    reconstructions: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    beta: float,
) -> torch.Tensor:
    return reconstruction_mse(inputs, reconstructions) + beta * normalized_kl_divergence(mu, logvar)


def svdd_distance(embeddings: torch.Tensor, center: torch.Tensor) -> torch.Tensor:
    return torch.sum((embeddings - center.unsqueeze(0)).pow(2), dim=1)


def absolute_error_map(inputs: torch.Tensor, reconstructions: torch.Tensor) -> torch.Tensor:
    return torch.abs(reconstructions - inputs)


def squared_error_map(inputs: torch.Tensor, reconstructions: torch.Tensor) -> torch.Tensor:
    return (reconstructions - inputs).pow(2)


def spatial_mean(error_map: torch.Tensor) -> torch.Tensor:
    dims = tuple(range(1, error_map.ndim))
    return error_map.mean(dim=dims)


def spatial_max(error_map: torch.Tensor) -> torch.Tensor:
    flattened = error_map.flatten(start_dim=1)
    return flattened.max(dim=1).values


def topk_spatial_mean(error_map: torch.Tensor, topk_ratio: float) -> torch.Tensor:
    if not 0.0 < topk_ratio <= 1.0:
        raise ValueError(f"topk_ratio must be in (0, 1], got {topk_ratio}")

    flattened = error_map.flatten(start_dim=1)
    topk = max(1, int(math.ceil(flattened.shape[1] * topk_ratio)))
    return torch.topk(flattened, k=topk, dim=1).values.mean(dim=1)


def masked_spatial_mean(error_map: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    if error_map.shape != mask.shape:
        raise ValueError(f"mask shape {tuple(mask.shape)} must match error_map shape {tuple(error_map.shape)}")

    flattened_error = error_map.flatten(start_dim=1)
    flattened_mask = mask.flatten(start_dim=1).to(dtype=error_map.dtype)
    weighted_sum = (flattened_error * flattened_mask).sum(dim=1)
    denominator = flattened_mask.sum(dim=1).clamp_min(1.0)
    return weighted_sum / denominator


def pooled_error_map(error_map: torch.Tensor, kernel_size: int) -> torch.Tensor:
    if kernel_size <= 0 or kernel_size % 2 == 0:
        raise ValueError(f"kernel_size must be a positive odd integer, got {kernel_size}")
    padding = kernel_size // 2
    return F.avg_pool2d(error_map, kernel_size=kernel_size, stride=1, padding=padding)

from dataclasses import dataclass

"""Shared training datatypes used by model-specific training loops."""
# src/wafer_defect/training/common.py


from dataclasses import dataclass


@dataclass
class EpochMetrics:
    loss: float
    reconstruction_loss: float
    kl_loss: float = 0.0
    distillation_loss: float = 0.0
    auxiliary_loss: float = 0.0


def _build_resnet_backbone(backbone_name: str, pretrained: bool) -> nn.Module:
    backbone_name = backbone_name.lower()
    if backbone_name != "resnet50":
        raise ValueError(f"Self-contained Modal notebook only supports resnet50, got {backbone_name}")
    weights = ResNet50_Weights.DEFAULT if pretrained else None
    return resnet50(weights=weights)


class ResNetFeatureExtractor(nn.Module):
    def __init__(
        self,
        backbone_name: str = "resnet50",
        pretrained: bool = True,
        input_size: int = 224,
        freeze_backbone: bool = True,
        normalize_imagenet: bool = True,
    ) -> None:
        super().__init__()
        backbone = _build_resnet_backbone(backbone_name, pretrained=pretrained)
        original_conv = backbone.conv1
        adapted_conv = nn.Conv2d(
            1,
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            adapted_conv.weight.copy_(original_conv.weight.mean(dim=1, keepdim=True))
        backbone.conv1 = adapted_conv

        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = backbone.avgpool
        self.embedding_dim = backbone.fc.in_features
        self.input_size = int(input_size)
        self.normalize_imagenet = bool(normalize_imagenet)
        self.register_buffer("image_mean", torch.tensor([0.4490], dtype=torch.float32).view(1, 1, 1, 1))
        self.register_buffer("image_std", torch.tensor([0.2260], dtype=torch.float32).view(1, 1, 1, 1))

        if freeze_backbone:
            for parameter in self.parameters():
                parameter.requires_grad = False

    def preprocess(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.input_size or x.shape[-2] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        if self.normalize_imagenet:
            x = (x - self.image_mean) / self.image_std
        return x

    def forward_intermediate_feature_map(self, x: torch.Tensor, layer_name: str = "layer4") -> torch.Tensor:
        layer_name = layer_name.lower()
        x = self.preprocess(x)
        x = self.stem(x)
        x = self.layer1(x)
        if layer_name == "layer1":
            return x
        x = self.layer2(x)
        if layer_name == "layer2":
            return x
        x = self.layer3(x)
        if layer_name == "layer3":
            return x
        return self.layer4(x)


"""Teacher-student distillation anomaly detector for wafer maps."""


import torch
from torch import nn
from torch.nn import functional as F



def _teacher_feature_dim(backbone_name: str, layer_name: str) -> int:
    backbone_name = backbone_name.lower()
    layer_name = layer_name.lower()

    dims = {
        "resnet18": {"layer1": 64, "layer2": 128, "layer3": 256, "layer4": 512},
        "resnet50": {"layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048},
    }
    if backbone_name not in dims or layer_name not in dims[backbone_name]:
        raise ValueError(f"Unsupported teacher backbone/layer combination: {backbone_name} / {layer_name}")
    return dims[backbone_name][layer_name]


class TSStudent(nn.Module):
    def __init__(self, feature_dim: int, input_size: int = 224) -> None:
        super().__init__()
        if input_size % 8 != 0:
            raise ValueError(f"input_size must be divisible by 8, got {input_size}")

        self.input_size = input_size
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, feature_dim, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim, feature_dim, kernel_size=3, stride=1, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


class TeacherFeatureAutoencoder(nn.Module):
    def __init__(self, feature_dim: int, hidden_dim: int = 64) -> None:
        super().__init__()
        if hidden_dim <= 0:
            raise ValueError(f"hidden_dim must be positive, got {hidden_dim}")

        bottleneck_dim = max(hidden_dim // 2, 16)
        self.encoder = nn.Sequential(
            nn.Conv2d(feature_dim, hidden_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, bottleneck_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(bottleneck_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden_dim, feature_dim, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


class TSDistillationModel(nn.Module):
    def __init__(
        self,
        image_size: int = 64,
        teacher_backbone: str = "resnet18",
        teacher_layer: str = "layer2",
        teacher_pretrained: bool = True,
        teacher_input_size: int = 224,
        normalize_teacher_input: bool = True,
        reduction: str = "topk_mean",
        topk_ratio: float = 0.01,
        feature_autoencoder_hidden_dim: int = 64,
        student_weight: float = 1.0,
        autoencoder_weight: float = 1.0,
        score_student_weight: float | None = None,
        score_autoencoder_weight: float | None = None,
    ) -> None:
        super().__init__()
        if reduction not in {"max", "mean", "topk_mean"}:
            raise ValueError(f"Unsupported reduction: {reduction}")
        if reduction == "topk_mean" and not 0.0 < topk_ratio <= 1.0:
            raise ValueError(f"topk_ratio must be in (0, 1], got {topk_ratio}")

        self.image_size = image_size
        self.teacher_backbone = teacher_backbone.lower()
        self.teacher_layer = teacher_layer.lower()
        self.teacher_input_size = int(teacher_input_size)
        self.reduction = reduction
        self.topk_ratio = float(topk_ratio)
        self.student_weight = float(student_weight)
        self.autoencoder_weight = float(autoencoder_weight)
        self.score_student_weight = (
            float(score_student_weight) if score_student_weight is not None else self.student_weight
        )
        self.score_autoencoder_weight = (
            float(score_autoencoder_weight) if score_autoencoder_weight is not None else self.autoencoder_weight
        )
        self.feature_dim = _teacher_feature_dim(self.teacher_backbone, self.teacher_layer)

        self.teacher = ResNetFeatureExtractor(
            backbone_name=self.teacher_backbone,
            pretrained=teacher_pretrained,
            input_size=self.teacher_input_size,
            freeze_backbone=True,
            normalize_imagenet=normalize_teacher_input,
        )
        self.student = TSStudent(feature_dim=self.feature_dim, input_size=self.teacher_input_size)
        self.autoencoder = TeacherFeatureAutoencoder(
            feature_dim=self.feature_dim,
            hidden_dim=int(feature_autoencoder_hidden_dim),
        )

        self.register_buffer("student_map_scale", torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer("autoencoder_map_scale", torch.tensor(1.0, dtype=torch.float32))

    def teacher_feature_map(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return self.teacher.forward_intermediate_feature_map(x, layer_name=self.teacher_layer)

    def _student_input(self, x: torch.Tensor) -> torch.Tensor:
        return self.teacher.preprocess(x)

    def student_feature_map(self, x: torch.Tensor) -> torch.Tensor:
        return self.student(self._student_input(x))

    def autoencoder_feature_map(self, teacher_features: torch.Tensor) -> torch.Tensor:
        autoencoder_features = self.autoencoder(teacher_features)
        if autoencoder_features.shape[-2:] != teacher_features.shape[-2:]:
            autoencoder_features = F.interpolate(
                autoencoder_features,
                size=teacher_features.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        return autoencoder_features

    def raw_anomaly_maps(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        teacher_features = self.teacher_feature_map(x)
        student_features = self.student_feature_map(x)
        if student_features.shape[-2:] != teacher_features.shape[-2:]:
            # Allow layer sweeps across teacher stages with different spatial resolutions.
            student_features = F.interpolate(
                student_features,
                size=teacher_features.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        autoencoder_features = self.autoencoder_feature_map(teacher_features)
        student_map = torch.mean((student_features - teacher_features).pow(2), dim=1, keepdim=True)
        autoencoder_map = torch.mean((autoencoder_features - teacher_features).pow(2), dim=1, keepdim=True)
        return student_map, autoencoder_map

    def normalized_anomaly_map(self, x: torch.Tensor) -> torch.Tensor:
        student_map, autoencoder_map = self.raw_anomaly_maps(x)
        student_scale = self.student_map_scale.clamp_min(1e-6)
        autoencoder_scale = self.autoencoder_map_scale.clamp_min(1e-6)
        combined = (
            self.score_student_weight * (student_map / student_scale)
            + self.score_autoencoder_weight * (autoencoder_map / autoencoder_scale)
        )
        return combined

    def set_error_scales(self, student_scale: float, autoencoder_scale: float) -> None:
        self.student_map_scale.fill_(max(float(student_scale), 1e-6))
        self.autoencoder_map_scale.fill_(max(float(autoencoder_scale), 1e-6))

    def reduce_anomaly_map(self, anomaly_map: torch.Tensor) -> torch.Tensor:
        if self.reduction == "mean":
            return spatial_mean(anomaly_map)
        if self.reduction == "max":
            return spatial_max(anomaly_map)
        return topk_spatial_mean(anomaly_map, topk_ratio=self.topk_ratio)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        anomaly_map = self.normalized_anomaly_map(x)
        return self.reduce_anomaly_map(anomaly_map)


def build_ts_distillation_from_config(config: dict, image_size: int) -> TSDistillationModel:
    model_config = config.get("model", {})
    return TSDistillationModel(
        image_size=image_size,
        teacher_backbone=str(model_config.get("teacher_backbone", "resnet18")),
        teacher_layer=str(model_config.get("teacher_layer", "layer2")),
        teacher_pretrained=bool(model_config.get("teacher_pretrained", True)),
        teacher_input_size=int(model_config.get("teacher_input_size", 224)),
        normalize_teacher_input=bool(model_config.get("normalize_teacher_input", True)),
        reduction=str(model_config.get("reduction", "topk_mean")),
        topk_ratio=float(model_config.get("topk_ratio", 0.01)),
        feature_autoencoder_hidden_dim=int(model_config.get("feature_autoencoder_hidden_dim", 64)),
        student_weight=float(model_config.get("student_weight", 1.0)),
        autoencoder_weight=float(model_config.get("autoencoder_weight", 1.0)),
        score_student_weight=model_config.get("score_student_weight"),
        score_autoencoder_weight=model_config.get("score_autoencoder_weight"),
    )

"""Training helpers for teacher-student distillation anomaly detection."""
#src/wafer_defect/training/ts_distillation.py


import torch
from torch import nn
from torch.utils.data import DataLoader



def run_ts_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
    desc: str | None = None,
) -> EpochMetrics:
    is_training = optimizer is not None
    model.train(is_training)
    if hasattr(model, "teacher"):
        model.teacher.eval()

    total_loss = 0.0
    total_student_loss = 0.0
    total_autoencoder_loss = 0.0
    total_items = 0

    iterator = dataloader
    if desc is not None:
        iterator = tqdm(dataloader, desc=desc, leave=False)

    for inputs, labels in iterator:
        inputs = inputs.to(device)
        labels = labels.to(device)

        normal_mask = labels == 0
        if not torch.any(normal_mask):
            continue

        normal_inputs = inputs[normal_mask]
        student_map, autoencoder_map = model.raw_anomaly_maps(normal_inputs)
        student_loss = student_map.mean()
        autoencoder_loss = autoencoder_map.mean()
        loss = model.student_weight * student_loss + model.autoencoder_weight * autoencoder_loss

        if is_training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        batch_size = normal_inputs.shape[0]
        total_loss += loss.item() * batch_size
        total_student_loss += student_loss.item() * batch_size
        total_autoencoder_loss += autoencoder_loss.item() * batch_size
        total_items += batch_size
        if desc is not None:
            iterator.set_postfix(
                loss=f"{(total_loss / max(total_items, 1)):.4f}",
                distill=f"{(total_student_loss / max(total_items, 1)):.4f}",
                feat_ae=f"{(total_autoencoder_loss / max(total_items, 1)):.4f}",
            )

    average_loss = total_loss / max(total_items, 1)
    average_student_loss = total_student_loss / max(total_items, 1)
    average_autoencoder_loss = total_autoencoder_loss / max(total_items, 1)
    return EpochMetrics(
        loss=average_loss,
        reconstruction_loss=average_student_loss,
        kl_loss=average_autoencoder_loss,
        distillation_loss=average_student_loss,
        auxiliary_loss=average_autoencoder_loss,
    )


def estimate_ts_error_scales(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()

    total_student = 0.0
    total_autoencoder = 0.0
    total_items = 0

    with torch.inference_mode():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            normal_mask = labels == 0
            if not torch.any(normal_mask):
                continue

            student_map, autoencoder_map = model.raw_anomaly_maps(inputs[normal_mask])
            batch_size = int(normal_mask.sum().item())
            total_student += float(student_map.mean().item()) * batch_size
            total_autoencoder += float(autoencoder_map.mean().item()) * batch_size
            total_items += batch_size

    if total_items == 0:
        raise ValueError("Could not estimate TS distillation error scales because no normal samples were found.")

    student_scale = total_student / total_items
    autoencoder_scale = total_autoencoder / total_items
    return max(student_scale, 1e-6), max(autoencoder_scale, 1e-6)

"""Metric helpers for reconstruction-based anomaly detection evaluation."""
# src/wafer_defect/evaluation/reconstruction_metrics.py


from typing import Any

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)


def summarize_threshold_metrics(
    labels: np.ndarray,
    scores: np.ndarray,
    threshold: float,
) -> dict[str, Any]:
    predicted = (scores > threshold).astype(int)
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "auroc": float(roc_auc_score(labels, scores)),
        "auprc": float(average_precision_score(labels, scores)),
        "predicted_anomalies": int(predicted.sum()),
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
    }


def sweep_threshold_metrics(labels: np.ndarray, scores: np.ndarray) -> tuple[pd.DataFrame, dict[str, Any]]:
    precision_curve, recall_curve, thresholds = precision_recall_curve(labels, scores)

    if thresholds.size == 0:
        sweep_df = pd.DataFrame(
            [
                {
                    "threshold": float(scores[0]) if scores.size else 0.0,
                    "precision": float(precision_curve[0]),
                    "recall": float(recall_curve[0]),
                    "f1": 0.0,
                    "predicted_anomalies": int((scores > 0).sum()),
                }
            ]
        )
    else:
        sweep_df = pd.DataFrame(
            {
                "threshold": thresholds,
                "precision": precision_curve[:-1],
                "recall": recall_curve[:-1],
            }
        )
        sweep_df["f1"] = (
            2
            * sweep_df["precision"]
            * sweep_df["recall"]
            / (sweep_df["precision"] + sweep_df["recall"] + 1e-12)
        )
        sweep_df["predicted_anomalies"] = [(scores > threshold).sum() for threshold in sweep_df["threshold"]]

    best_row = sweep_df.loc[sweep_df["f1"].idxmax()]
    best_summary = {
        "threshold": float(best_row["threshold"]),
        "precision": float(best_row["precision"]),
        "recall": float(best_row["recall"]),
        "f1": float(best_row["f1"]),
        "predicted_anomalies": int(best_row["predicted_anomalies"]),
    }
    return sweep_df, best_summary


def clone_state_dict(model: torch.nn.Module) -> dict[str, torch.Tensor]:
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def collect_raw_maps(model: nn.Module, dataloader: DataLoader, device: torch.device, desc: str) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    student_maps: list[np.ndarray] = []
    autoencoder_maps: list[np.ndarray] = []
    with torch.inference_mode():
        for inputs, _ in tqdm(dataloader, desc=desc, leave=False):
            inputs = inputs.to(device, non_blocking=True)
            student_map, autoencoder_map = model.raw_anomaly_maps(inputs)
            student_maps.append(student_map.cpu().numpy())
            autoencoder_maps.append(autoencoder_map.cpu().numpy())
    return np.concatenate(student_maps, axis=0), np.concatenate(autoencoder_maps, axis=0)


def reduce_error_map_numpy(error_map: np.ndarray, reduction: str, topk_ratio: float) -> np.ndarray:
    error_map = np.nan_to_num(error_map.astype(np.float32), nan=0.0, posinf=1e6, neginf=0.0)
    flat = error_map.reshape(error_map.shape[0], -1)
    if reduction == "mean":
        return flat.mean(axis=1)
    if reduction == "max":
        return flat.max(axis=1)
    topk = max(1, int(math.ceil(flat.shape[1] * topk_ratio)))
    partitioned = np.partition(flat, kth=flat.shape[1] - topk, axis=1)
    return partitioned[:, -topk:].mean(axis=1)


def build_defect_breakdown(test_metadata: pd.DataFrame, scores: np.ndarray, threshold: float) -> pd.DataFrame:
    anomaly_df = test_metadata[test_metadata["is_anomaly"] == 1].reset_index(drop=True).copy()
    anomaly_df["score"] = scores[test_metadata["is_anomaly"].to_numpy(dtype=bool)]
    anomaly_df["predicted_anomaly"] = (anomaly_df["score"].to_numpy() > threshold).astype(int)
    rows: list[dict[str, Any]] = []
    for defect_type, defect_rows in anomaly_df.groupby("defect_type"):
        count = int(len(defect_rows))
        detected = int(defect_rows["predicted_anomaly"].sum())
        rows.append(
            {
                "defect_type": defect_type,
                "count": count,
                "detected": detected,
                "mean_score": float(defect_rows["score"].mean()),
                "median_score": float(defect_rows["score"].median()),
                "missed": count - detected,
                "recall": float(detected / max(count, 1)),
            }
        )
    return pd.DataFrame(rows).sort_values(["recall", "count", "defect_type"], ascending=[True, False, True]).reset_index(drop=True)


def evaluate_score_variants(
    model: nn.Module,
    val_student_maps: np.ndarray,
    val_auto_maps: np.ndarray,
    test_student_maps: np.ndarray,
    test_auto_maps: np.ndarray,
    val_metadata: pd.DataFrame,
    test_metadata: pd.DataFrame,
    config: dict[str, Any],
) -> dict[str, Any]:
    score_rows: list[dict[str, Any]] = []
    best_payload: dict[str, Any] | None = None
    progress = tqdm(config["scoring"]["variants"], desc="score_variants", leave=True)
    val_labels = val_metadata["is_anomaly"].to_numpy(dtype=int)
    test_labels = test_metadata["is_anomaly"].to_numpy(dtype=int)

    for variant in progress:
        val_map = (
            float(variant["score_student_weight"]) * (val_student_maps / float(model.student_map_scale.item()))
            + float(variant["score_autoencoder_weight"]) * (val_auto_maps / float(model.autoencoder_map_scale.item()))
        )
        test_map = (
            float(variant["score_student_weight"]) * (test_student_maps / float(model.student_map_scale.item()))
            + float(variant["score_autoencoder_weight"]) * (test_auto_maps / float(model.autoencoder_map_scale.item()))
        )
        val_scores = reduce_error_map_numpy(val_map, str(variant["reduction"]), float(variant["topk_ratio"]))
        test_scores = reduce_error_map_numpy(test_map, str(variant["reduction"]), float(variant["topk_ratio"]))
        threshold = float(np.quantile(val_scores[val_labels == 0], float(config["scoring"]["threshold_quantile"])))
        metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
        row = {
            "variant": str(variant["name"]),
            "reduction": str(variant["reduction"]),
            "topk_ratio": float(variant["topk_ratio"]),
            "score_student_weight": float(variant["score_student_weight"]),
            "score_autoencoder_weight": float(variant["score_autoencoder_weight"]),
            **metrics,
        }
        score_rows.append(row)
        progress.set_postfix(f1=f"{metrics['f1']:.4f}", recall=f"{metrics['recall']:.4f}")
        if best_payload is None or row["f1"] > best_payload["best_row"]["f1"]:
            threshold_sweep_df, best_threshold_sweep = sweep_threshold_metrics(test_labels, test_scores)
            best_payload = {
                "best_row": row,
                "threshold_sweep_df": threshold_sweep_df,
                "best_threshold_sweep": best_threshold_sweep,
                "best_val_scores": val_scores,
                "best_test_scores": test_scores,
            }

    if best_payload is None:
        raise RuntimeError("No score variants were evaluated.")
    score_df = pd.DataFrame(score_rows).sort_values(["f1", "auprc", "recall"], ascending=False).reset_index(drop=True)
    defect_breakdown_df = build_defect_breakdown(test_metadata, best_payload["best_test_scores"], best_payload["best_row"]["threshold"])
    return {
        "score_df": score_df,
        "best_row": best_payload["best_row"],
        "threshold_sweep_df": best_payload["threshold_sweep_df"],
        "best_threshold_sweep": best_payload["best_threshold_sweep"],
        "best_val_scores": best_payload["best_val_scores"],
        "best_test_scores": best_payload["best_test_scores"],
        "defect_breakdown_df": defect_breakdown_df,
    }


def train_ts_model(metadata: pd.DataFrame, config: dict[str, Any], output_dir: Path, device: torch.device) -> dict[str, Any]:
    train_dataset = PreparedWaferDataset(metadata, split="train")
    val_dataset = PreparedWaferDataset(metadata, split="val")
    batch_size = int(config["data"]["batch_size"])
    num_workers = int(config["data"]["num_workers"])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=device.type == "cuda")
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=device.type == "cuda")

    model = build_ts_distillation_from_config(config, image_size=int(config["data"]["image_size"])).to(device)
    optimizer = torch.optim.Adam(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=float(config["training"]["learning_rate"]),
        weight_decay=float(config["training"]["weight_decay"]),
    )

    history: list[dict[str, float]] = []
    best_val_loss = float("inf")
    best_epoch = 0
    best_state_dict: dict[str, torch.Tensor] | None = None
    stale_epochs = 0
    start_epoch = 0
    latest_checkpoint_path = output_dir / "latest_checkpoint.pt"

    if bool(config["run"].get("resume_if_available", True)) and latest_checkpoint_path.exists():
        checkpoint = torch.load(latest_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        history = list(checkpoint.get("history", []))
        start_epoch = int(checkpoint.get("epoch", 0))
        best_val_loss = float(checkpoint.get("best_val_loss", best_val_loss))
        best_epoch = int(checkpoint.get("best_epoch", 0))
        stale_epochs = int(checkpoint.get("stale_epochs", 0))
        print(f"Resumed from {latest_checkpoint_path} at epoch {start_epoch}")

    epoch_iterator = tqdm(range(start_epoch + 1, int(config["training"]["epochs"]) + 1), desc="epochs", leave=True)
    for epoch in epoch_iterator:
        train_metrics = run_ts_epoch(model, train_loader, device, optimizer=optimizer, desc=f"train:e{epoch}")
        val_metrics = run_ts_epoch(model, val_loader, device, desc=f"val:e{epoch}")
        record = {
            "epoch": epoch,
            "train_loss": train_metrics.loss,
            "train_distillation_loss": train_metrics.distillation_loss,
            "train_autoencoder_loss": train_metrics.auxiliary_loss,
            "val_loss": val_metrics.loss,
            "val_distillation_loss": val_metrics.distillation_loss,
            "val_autoencoder_loss": val_metrics.auxiliary_loss,
        }
        history.append(record)
        epoch_iterator.set_postfix(train=f"{train_metrics.loss:.4f}", val=f"{val_metrics.loss:.4f}", best=f"{best_val_loss:.4f}")

        improved = (best_val_loss - val_metrics.loss) > float(config["training"].get("early_stopping_min_delta", 0.0))
        if improved:
            best_val_loss = float(val_metrics.loss)
            best_epoch = epoch
            best_state_dict = clone_state_dict(model)
            stale_epochs = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": best_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                    "best_epoch": best_epoch,
                    "best_val_loss": best_val_loss,
                    "stale_epochs": stale_epochs,
                    "history": history,
                },
                output_dir / "best_model.pt",
            )
        else:
            stale_epochs += 1

        latest_checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "stale_epochs": stale_epochs,
            "history": history,
        }
        torch.save(latest_checkpoint, latest_checkpoint_path)
        checkpoint_every = int(config["training"].get("checkpoint_every", 0))
        if checkpoint_every > 0 and epoch % checkpoint_every == 0:
            torch.save(latest_checkpoint, output_dir / f"checkpoint_epoch_{epoch}.pt")
        patience = int(config["training"].get("early_stopping_patience", 0))
        if patience > 0 and stale_epochs >= patience:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}, best val loss: {best_val_loss:.6f}")
            break

    if best_state_dict is None:
        best_state_dict = clone_state_dict(model)

    model.load_state_dict(best_state_dict)
    student_scale, autoencoder_scale = estimate_ts_error_scales(model, train_loader, device)
    model.set_error_scales(student_scale, autoencoder_scale)
    calibrated_state = clone_state_dict(model)

    torch.save(
        {
            "epoch": best_epoch,
            "model_state_dict": calibrated_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "stale_epochs": stale_epochs,
            "history": history,
            "student_map_scale": student_scale,
            "autoencoder_map_scale": autoencoder_scale,
        },
        output_dir / "best_model.pt",
    )
    torch.save(
        {
            "epoch": len(history),
            "model_state_dict": calibrated_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "stale_epochs": stale_epochs,
            "history": history,
            "student_map_scale": student_scale,
            "autoencoder_map_scale": autoencoder_scale,
        },
        output_dir / "last_model.pt",
    )

    history_df = pd.DataFrame(history)
    history_df.to_json(output_dir / "history.json", orient="records", indent=2)
    history_df.to_csv(output_dir / "history.csv", index=False)
    summary = {
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "epochs_ran": int(len(history)),
        "student_map_scale": float(student_scale),
        "autoencoder_map_scale": float(autoencoder_scale),
        "output_dir": str(output_dir),
    }
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))
    return {"model": model, "history_df": history_df, "summary": summary}


In [ ]:
metadata = prepare_dataset(CONFIG, rebuild=bool(CONFIG["run"].get("rebuild_dataset", False)))
display(metadata.head())
display(metadata.groupby(["split", "is_anomaly"]).size().rename("count").reset_index())
print(f"Metadata path: {METADATA_PATH}")
print(f"Arrays dir: {ARRAYS_DIR}")


In [ ]:
training_result = None
if bool(CONFIG["run"].get("run_training", True)):
    training_result = train_ts_model(metadata, CONFIG, OUTPUT_DIR, DEVICE)
else:
    checkpoint = torch.load(OUTPUT_DIR / "best_model.pt", map_location=DEVICE)
    model = build_ts_distillation_from_config(checkpoint["config"], image_size=int(checkpoint["config"]["data"]["image_size"])).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    training_result = {
        "model": model,
        "history_df": pd.read_csv(OUTPUT_DIR / "history.csv"),
        "summary": json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8")),
    }

display(training_result["history_df"].tail())
history_df = training_result["history_df"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val")
axes[0].set_title("TS Total Loss")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["train_distillation_loss"], marker="o", label="train distill")
axes[1].plot(history_df["epoch"], history_df["val_distillation_loss"], marker="o", label="val distill")
axes[1].plot(history_df["epoch"], history_df["train_autoencoder_loss"], marker="o", linestyle="--", label="train feat AE")
axes[1].plot(history_df["epoch"], history_df["val_autoencoder_loss"], marker="o", linestyle="--", label="val feat AE")
axes[1].set_title("Branch Losses")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
model = training_result["model"]
batch_size = int(CONFIG["data"]["batch_size"])
num_workers = int(CONFIG["data"]["num_workers"])
val_dataset = PreparedWaferDataset(metadata, split="val")
test_dataset = PreparedWaferDataset(metadata, split="test")
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=DEVICE.type == "cuda")
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=DEVICE.type == "cuda")

val_student_maps, val_auto_maps = collect_raw_maps(model, val_loader, DEVICE, desc="maps_val")
test_student_maps, test_auto_maps = collect_raw_maps(model, test_loader, DEVICE, desc="maps_test")

val_metadata = metadata[metadata["split"] == "val"].reset_index(drop=True).copy()
test_metadata = metadata[metadata["split"] == "test"].reset_index(drop=True).copy()

evaluation_result = evaluate_score_variants(
    model,
    val_student_maps,
    val_auto_maps,
    test_student_maps,
    test_auto_maps,
    val_metadata,
    test_metadata,
    CONFIG,
)

evaluation_result["score_df"].to_csv(EVAL_DIR / "score_sweep.csv", index=False)
evaluation_result["threshold_sweep_df"].to_csv(EVAL_DIR / "threshold_sweep.csv", index=False)
evaluation_result["defect_breakdown_df"].to_csv(EVAL_DIR / "best_defect_breakdown.csv", index=False)

best_test_scores_df = test_metadata.copy()
best_test_scores_df["score"] = evaluation_result["best_test_scores"]
best_test_scores_df["predicted_anomaly"] = (best_test_scores_df["score"] > evaluation_result["best_row"]["threshold"]).astype(int)
best_test_scores_df.to_csv(EVAL_DIR / "best_test_scores.csv", index=False)

best_val_scores_df = val_metadata.copy()
best_val_scores_df["score"] = evaluation_result["best_val_scores"]
best_val_scores_df.to_csv(EVAL_DIR / "best_val_scores.csv", index=False)

summary_payload = {
    "training": training_result["summary"],
    "best_score_variant": evaluation_result["best_row"],
    "best_threshold_sweep": evaluation_result["best_threshold_sweep"],
}
(EVAL_DIR / "summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print("Best score variant:")
print(json.dumps(evaluation_result["best_row"], indent=2))
display(evaluation_result["score_df"])
print("Hardest defect types:")
display(evaluation_result["defect_breakdown_df"].head(12))
print(f"Saved outputs to {OUTPUT_DIR}")


In [ ]:
summary_files = {
    "history_csv": OUTPUT_DIR / "history.csv",
    "best_model": OUTPUT_DIR / "best_model.pt",
    "last_model": OUTPUT_DIR / "last_model.pt",
    "training_summary": OUTPUT_DIR / "summary.json",
    "score_sweep": EVAL_DIR / "score_sweep.csv",
    "threshold_sweep": EVAL_DIR / "threshold_sweep.csv",
    "best_val_scores": EVAL_DIR / "best_val_scores.csv",
    "best_test_scores": EVAL_DIR / "best_test_scores.csv",
    "best_defect_breakdown": EVAL_DIR / "best_defect_breakdown.csv",
    "eval_summary": EVAL_DIR / "summary.json",
}
for name, path in summary_files.items():
    print(f"{name}: {path} | exists={path.exists()}")
